# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ghayoorahmed7/flyrank_ml_intership/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


1. My rule and its reason codes

Before writing the rule, I checked the two signals it leans on.

**Signal A — staleness**, behind the refresh flags. Assumption: content that hasn't been
updated in a long time performs worse than fresh content.

**Signal B — CTR-vs-position**, behind the CTR-fix logic. Assumption: CTR should track ranking
position, so CTR far below the norm for a position is a real problem, not noise.

Per `flyrank-data`: rate columns (`ctr`, `engagement_rate`) are already ×100 percentages, and
`avg_position == 0` means *no data*, not rank zero — excluded from every check below.
`trend_direction` / `trend_pct` are label-derived and never used as inputs.

In [16]:
import pandas as pd
import numpy as np

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 60)

df = pd.read_csv('/content/content_refresh_anonymized.csv')
print(df.shape)
df.head(3)

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


### Re-running Signal A: Staleness vs. Performance

In [23]:
# --- Signal A: staleness vs performance, on content that's still visible ---
visible = df[df['impressions_90d'] >= 500].copy()
visible = visible[visible['avg_position'] > 0]

tier_order = ['0-30', '31-90', '91-180', '181+']
signal_a = visible.groupby('freshness_tier').agg(
    n=('content_id', 'size'),
    mean_ctr=('ctr', 'mean'),
    median_ctr=('ctr', 'median'),
    mean_position=('avg_position', 'mean'),
    median_position=('avg_position', 'median'),
).reindex(tier_order).round(3)

print("Signal A — staleness vs performance, visible content only")
signal_a

Signal A — staleness vs performance, visible content only


,n,mean_ctr,median_ctr,mean_position,median_position
freshness_tier,,,,,
0-30,10063,0.271,0.18,15.070,10.6
31-90,88,0.145,0.10,16.475,12.9
91-180,6558,0.250,0.15,16.869,13.1
181+,17,0.209,0.20,20.671,18.6


### Re-running Signal B: CTR should track position

In [24]:
# --- Signal B: CTR should track position ---
has_pos = df[df['avg_position'] > 0].copy()

pos_order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
signal_b = has_pos.groupby('position_tier').agg(
    n=('content_id', 'size'),
    mean_ctr=('ctr', 'mean'),
    median_ctr=('ctr', 'median'),
    mean_position=('avg_position', 'mean'),
).reindex(pos_order).round(3)

print("Signal B — CTR by position tier")
signal_b

Signal B — CTR by position tier


,n,mean_ctr,median_ctr,mean_position
position_tier,,,,
top_3,1116,2.764,0.00,2.102
page_1,11814,0.652,0.16,6.573
striking,7304,0.323,0.11,14.260
page_3_5,7242,0.222,0.03,30.674
deep,1319,0.150,0.00,63.665


### Building the Ranked Queue

In [25]:
expected_ctr_by_tier = has_pos.groupby('position_tier')['ctr'].mean()

work = df.copy()
work['has_position_data'] = work['avg_position'] > 0
work['expected_ctr_for_tier'] = work['position_tier'].map(expected_ctr_by_tier)
work['underperform_ratio'] = (work['ctr'] / work['expected_ctr_for_tier']).round(3)

is_stale = work['freshness_tier'].isin(['91-180', '181+'])
is_visible = work['impressions_90d'] >= 500
is_underperforming = work['underperform_ratio'] < 0.5
is_flagged = is_stale & is_visible & is_underperforming & work['has_position_data']

work['reason_code'] = 'stale_underperforming_ctr_for_position'
work['score'] = 0
work.loc[is_flagged, 'score'] = work.loc[is_flagged, 'impressions_90d']
work['action'] = np.where(work['score'] > 0, 'refresh', 'no_action')

print("flagged:", int(is_flagged.sum()), "of", len(work), f"({is_flagged.mean():.1%})")
work[['action', 'reason_code']].value_counts()

flagged: 3966 of 30000 (13.2%)


,,count
action,reason_code,
no_action,stale_underperforming_ctr_for_position,26034
refresh,stale_underperforming_ctr_for_position,3966


In [26]:
queue = work.sort_values('score', ascending=False).reset_index(drop=True)

output_cols = ['content_id', 'client_id', 'score', 'action', 'reason_code',
               'freshness_tier', 'days_since_last_update', 'impressions_90d', 'clicks_90d',
               'ctr', 'expected_ctr_for_tier', 'underperform_ratio', 'avg_position',
               'position_tier', 'search_volume']

import os
os.makedirs('work/outputs', exist_ok=True)
queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("wrote work/outputs/baseline_action_score.csv —", len(queue), "rows")
queue[output_cols].head(3)

wrote work/outputs/baseline_action_score.csv — 30000 rows


,content_id,client_id,score,action,reason_code,freshness_tier,days_since_last_update,impressions_90d,clicks_90d,ctr,expected_ctr_for_tier,underperform_ratio,avg_position,position_tier,search_volume
0,content_5fe46e04994d,client_4e07408562,517715,refresh,stale_underperforming_ctr_for_position,91-180,104,517715,741,0.14,0.652467,0.215,4.2,page_1,1900.0
1,content_cb112fce36be,client_19581e27de,309910,refresh,stale_underperforming_ctr_for_position,91-180,104,309910,492,0.16,0.652467,0.245,5.6,page_1,70.0
2,content_9532f197bbc8,client_4e07408562,309192,refresh,stale_underperforming_ctr_for_position,91-180,104,309192,2689,0.87,2.764453,0.315,2.0,top_3,10.0


### Displaying the Top 20

In [27]:
top20 = queue[output_cols].head(20).reset_index(drop=True)
top20.index = top20.index + 1
top20

,content_id,client_id,score,action,reason_code,freshness_tier,days_since_last_update,impressions_90d,clicks_90d,ctr,expected_ctr_for_tier,underperform_ratio,avg_position,position_tier,search_volume
1,content_5fe46e04994d,client_4e07408562,517715,refresh,stale_underperforming_ctr_for_position,91-180,104,517715,741,0.14,0.652467,0.215,4.2,page_1,1900.0
2,content_cb112fce36be,client_19581e27de,309910,refresh,stale_underperforming_ctr_for_position,91-180,104,309910,492,0.16,0.652467,0.245,5.6,page_1,70.0
3,content_9532f197bbc8,client_4e07408562,309192,refresh,stale_underperforming_ctr_for_position,91-180,104,309192,2689,0.87,2.764453,0.315,2.0,top_3,10.0
4,content_36ff89c8214e,client_19581e27de,295097,refresh,stale_underperforming_ctr_for_position,91-180,104,295097,154,0.05,0.652467,0.077,7.3,page_1,0.0
5,content_b28d1efd668f,client_6208ef0f77,286608,refresh,stale_underperforming_ctr_for_position,91-180,104,286608,169,0.06,0.222484,0.270,26.2,page_3_5,0.0
6,content_813e88069237,client_6208ef0f77,233561,refresh,stale_underperforming_ctr_for_position,91-180,104,233561,129,0.06,0.222484,0.270,26.2,page_3_5,0.0
7,content_c8e9d6ab9013,client_19581e27de,208678,refresh,stale_underperforming_ctr_for_position,91-180,104,208678,0,0.00,0.652467,0.000,9.7,page_1,20.0
8,content_d17681677e69,client_19581e27de,201584,refresh,stale_underperforming_ctr_for_position,91-180,104,201584,487,0.24,0.652467,0.368,5.8,page_1,10.0
9,content_a7427266c305,client_19581e27de,201111,refresh,stale_underperforming_ctr_for_position,91-180,104,201111,219,0.11,0.652467,0.169,5.7,page_1,0.0
10,content_3d94572c3a35,client_19581e27de,190623,refresh,stale_underperforming_ctr_for_position,91-180,104,190623,462,0.24,0.652467,0.368,4.3,page_1,50.0


In [17]:
# --- Signal A: staleness vs performance, on content that's still visible ---
visible = df[df['impressions_90d'] >= 500].copy()
visible = visible[visible['avg_position'] > 0]

tier_order = ['0-30', '31-90', '91-180', '181+']
signal_a = visible.groupby('freshness_tier').agg(
    n=('content_id', 'size'),
    mean_ctr=('ctr', 'mean'),
    median_ctr=('ctr', 'median'),
    mean_position=('avg_position', 'mean'),
    median_position=('avg_position', 'median'),
).reindex(tier_order).round(3)

print("Signal A — staleness vs performance, visible content only")
signal_a

Signal A — staleness vs performance, visible content only


,n,mean_ctr,median_ctr,mean_position,median_position
freshness_tier,,,,,
0-30,10063,0.271,0.18,15.070,10.6
31-90,88,0.145,0.10,16.475,12.9
91-180,6558,0.250,0.15,16.869,13.1
181+,17,0.209,0.20,20.671,18.6


**Verdict A: MIXED.**

Mean CTR does not fall monotonically with staleness (0.271 → 0.145 → 0.250 → 0.209). Average
position does drift worse as staleness increases (15.1 → 16.5 → 16.9 → 20.7) — directional but
weak, and the two oldest buckets are thin (n=88, n=17). `days_since_last_update` also clusters
heavily at exactly 20 and 104 days (11,573 and 8,773 rows) — likely a platform batch date, not
per-page editorial timing. Staleness alone is a weak, noisy gate — used below only as a
secondary condition, not the deciding one.

In [18]:
# --- Signal B: CTR should track position ---
has_pos = df[df['avg_position'] > 0].copy()

pos_order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
signal_b = has_pos.groupby('position_tier').agg(
    n=('content_id', 'size'),
    mean_ctr=('ctr', 'mean'),
    median_ctr=('ctr', 'median'),
    mean_position=('avg_position', 'mean'),
).reindex(pos_order).round(3)

print("Signal B — CTR by position tier")
signal_b

Signal B — CTR by position tier


,n,mean_ctr,median_ctr,mean_position
position_tier,,,,
top_3,1116,2.764,0.00,2.102
page_1,11814,0.652,0.16,6.573
striking,7304,0.323,0.11,14.260
page_3_5,7242,0.222,0.03,30.674
deep,1319,0.150,0.00,63.665


**Verdict B: CONFIRMED.**

Mean CTR falls monotonically as position worsens: 2.764 (top_3) → 0.652 (page_1) → 0.323
(striking) → 0.222 (page_3_5) → 0.150 (deep), all well-populated buckets (n=1,116–11,814).
`position_tier` mislabels all 1,205 `avg_position == 0` rows as `top_3` — filtered out above.

**My rule, in plain words:** A page is worth a refresh action if it still gets meaningful search
visibility, hasn't been touched in a long time, and its CTR is less than half of what pages at
its own position typically get.

- **stale** = `freshness_tier` in `{91-180, 181+}`
- **visible** = `impressions_90d >= 500`
- **has_position_data** = `avg_position > 0`
- **underperforming** = `ctr / expected_ctr_for_position_tier < 0.5`
- **score** = `impressions_90d` where all conditions hold, else `0`
- **reason_code** = `stale_underperforming_ctr_for_position` (one fixed code, every flagged row)
- **action** = `"refresh"` if `score > 0`, else `"no_action"`

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## 2. Build the ranked queue (writes the CSV)

In [19]:
expected_ctr_by_tier = has_pos.groupby('position_tier')['ctr'].mean()

work = df.copy()
work['has_position_data'] = work['avg_position'] > 0
work['expected_ctr_for_tier'] = work['position_tier'].map(expected_ctr_by_tier)
work['underperform_ratio'] = (work['ctr'] / work['expected_ctr_for_tier']).round(3)

is_stale = work['freshness_tier'].isin(['91-180', '181+'])
is_visible = work['impressions_90d'] >= 500
is_underperforming = work['underperform_ratio'] < 0.5
is_flagged = is_stale & is_visible & is_underperforming & work['has_position_data']

work['reason_code'] = 'stale_underperforming_ctr_for_position'
work['score'] = 0
work.loc[is_flagged, 'score'] = work.loc[is_flagged, 'impressions_90d']
work['action'] = np.where(work['score'] > 0, 'refresh', 'no_action')

print("flagged:", int(is_flagged.sum()), "of", len(work), f"({is_flagged.mean():.1%})")
work[['action', 'reason_code']].value_counts()

flagged: 3966 of 30000 (13.2%)


,,count
action,reason_code,
no_action,stale_underperforming_ctr_for_position,26034
refresh,stale_underperforming_ctr_for_position,3966


In [20]:
queue = work.sort_values('score', ascending=False).reset_index(drop=True)

output_cols = ['content_id', 'client_id', 'score', 'action', 'reason_code',
               'freshness_tier', 'days_since_last_update', 'impressions_90d', 'clicks_90d',
               'ctr', 'expected_ctr_for_tier', 'underperform_ratio', 'avg_position',
               'position_tier', 'search_volume']

import os
os.makedirs('work/outputs', exist_ok=True)
queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("wrote work/outputs/baseline_action_score.csv —", len(queue), "rows")
queue[output_cols].head(3)

wrote work/outputs/baseline_action_score.csv — 30000 rows


,content_id,client_id,score,action,reason_code,freshness_tier,days_since_last_update,impressions_90d,clicks_90d,ctr,expected_ctr_for_tier,underperform_ratio,avg_position,position_tier,search_volume
0,content_5fe46e04994d,client_4e07408562,517715,refresh,stale_underperforming_ctr_for_position,91-180,104,517715,741,0.14,0.652467,0.215,4.2,page_1,1900.0
1,content_cb112fce36be,client_19581e27de,309910,refresh,stale_underperforming_ctr_for_position,91-180,104,309910,492,0.16,0.652467,0.245,5.6,page_1,70.0
2,content_9532f197bbc8,client_4e07408562,309192,refresh,stale_underperforming_ctr_for_position,91-180,104,309192,2689,0.87,2.764453,0.315,2.0,top_3,10.0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

In [21]:
top20 = queue[output_cols].head(20).reset_index(drop=True)
top20.index = top20.index + 1
top20

,content_id,client_id,score,action,reason_code,freshness_tier,days_since_last_update,impressions_90d,clicks_90d,ctr,expected_ctr_for_tier,underperform_ratio,avg_position,position_tier,search_volume
1,content_5fe46e04994d,client_4e07408562,517715,refresh,stale_underperforming_ctr_for_position,91-180,104,517715,741,0.14,0.652467,0.215,4.2,page_1,1900.0
2,content_cb112fce36be,client_19581e27de,309910,refresh,stale_underperforming_ctr_for_position,91-180,104,309910,492,0.16,0.652467,0.245,5.6,page_1,70.0
3,content_9532f197bbc8,client_4e07408562,309192,refresh,stale_underperforming_ctr_for_position,91-180,104,309192,2689,0.87,2.764453,0.315,2.0,top_3,10.0
4,content_36ff89c8214e,client_19581e27de,295097,refresh,stale_underperforming_ctr_for_position,91-180,104,295097,154,0.05,0.652467,0.077,7.3,page_1,0.0
5,content_b28d1efd668f,client_6208ef0f77,286608,refresh,stale_underperforming_ctr_for_position,91-180,104,286608,169,0.06,0.222484,0.270,26.2,page_3_5,0.0
6,content_813e88069237,client_6208ef0f77,233561,refresh,stale_underperforming_ctr_for_position,91-180,104,233561,129,0.06,0.222484,0.270,26.2,page_3_5,0.0
7,content_c8e9d6ab9013,client_19581e27de,208678,refresh,stale_underperforming_ctr_for_position,91-180,104,208678,0,0.00,0.652467,0.000,9.7,page_1,20.0
8,content_d17681677e69,client_19581e27de,201584,refresh,stale_underperforming_ctr_for_position,91-180,104,201584,487,0.24,0.652467,0.368,5.8,page_1,10.0
9,content_a7427266c305,client_19581e27de,201111,refresh,stale_underperforming_ctr_for_position,91-180,104,201111,219,0.11,0.652467,0.169,5.7,page_1,0.0
10,content_3d94572c3a35,client_19581e27de,190623,refresh,stale_underperforming_ctr_for_position,91-180,104,190623,462,0.24,0.652467,0.368,4.3,page_1,50.0


**Rank 1** (517,715) — refresh; page_1 (pos 4.2), CTR 0.14 vs 0.652 expected — largest audience
in the data. *Wrong if:* a SERP feature above it is eating clicks a rewrite can't win back.

**Rank 2** (309,910) — refresh; same profile at smaller scale (ratio 0.245). *Wrong if:*
`search_volume` is only 70/mo — disproportionate to 309,910 impressions, worth checking scope.

**Rank 3** (309,192) — refresh; top_3 (pos 2.0), CTR 0.87 vs 2.764 expected. *Wrong if:* top_3's
median CTR is actually 0.00 (Signal B) — the mean baseline here is skewed by outliers.

**Rank 4** (295,097) — refresh; page_1 (pos 7.3), ratio 0.077 — worst ratio in the top 10, ranked
4th only because score rewards impressions. *Wrong if:* intent is informational and a SERP
feature already answers the query.

**Rank 5 & 6** (286,608 / 233,561) — refresh; same client, both page_3_5, identical
`avg_position = 26.2`. *Wrong if:* that repeated value is a client-level default, not two
independently measured positions.

**Rank 7** (208,678) — refresh; 0 clicks on 208,678 impressions. *Wrong if:* broken analytics
tag, not a content problem.

**Rank 8–10** (201,584 / 201,111 / 190,623) — refresh; all one client, page_1, ratios
0.169–0.368. *Wrong if:* this client is just large and stale-dated in bulk — ten rows, one
pattern.

**Rank 11** (181,514) — refresh; page_3_5 (pos 25.8), ratio 0.449. *Wrong if:* the click ceiling
at position ~26 is too small to justify a refresh regardless of ratio.

**Rank 12** (179,002) — refresh; page_3_5 (pos 22.1), same client as Rank 11. *Wrong if:* these
are two instances of one templated page type, not two independent judgments.

**Rank 13** (176,296) — refresh; page_1 (pos 4.3), commercial intent. *Wrong if:* commercial
queries at page_1 often see lower CTR from shopping/ad units — wrong peer group for comparison.

**Rank 14** (159,590) — refresh; page_1 (pos 7.8), ratio 0.092 — very severe. *Wrong if:*
`search_volume` is only 10/mo — a narrow-demand query.

**Rank 15** (152,617) — refresh; page_1 (pos 3.3), transactional intent. *Wrong if:* same
structural CTR-suppression concern as Rank 13.

**Rank 16** (152,467) — refresh; page_1 (pos 6.5), ratio 0.199. *Wrong if:* nothing distinct —
here on scale, not a unique problem.

**Rank 17** (151,541) — refresh; page_1 (pos 5.6), a third client. *Wrong if:* nothing specific —
useful check that the rule fires outside the two dominant clients.

**Rank 18** (149,083) — refresh; page_1 (pos 6.2), highest search volume in the top 20 (480).
*Wrong if:* the snippet is fine but a competitor's listing is simply more compelling.

**Rank 19** (147,670) — refresh; page_1 (pos 6.4), ratio 0.107. *Wrong if:* low ratio on modest
volume (40/mo) is hard to distinguish from query-level noise.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## 4. Weak picks + leakage check

**Weak picks:**
1. Rank 3 — judged against a mean top_3 CTR (2.764) that's skewed by outliers; median is 0.00.
2. Rank 7 — 0 clicks on 208,678 impressions could be a tracking bug, not a content problem.
3. Ranks 5 & 6 — identical avg_position (26.2) on two different pages, same client — verify it's
   not a shared default value.
4. All 20 rows share `days_since_last_update == 104`, and 17 of 20 come from just 2 clients —
   a snapshot-date artifact, not 20 independent editorial judgments.
5. `score = impressions_90d` ranks by audience reached, not by severity — Rank 4's ratio (0.077,
   worst in the top 10) sits below three less-severe, bigger-audience picks. Deliberate, but
   worth stating plainly.

**Leakage check:**
- No `trend_direction` / `trend_pct` anywhere in the rule or the expected-CTR baseline.
- No `*_last_30d` / `*_prev_30d` columns used as inputs.
- `content_id` / `client_id` used only for identification, never as features.
- `avg_position == 0` excluded explicitly, rather than trusting `position_tier`.
- No future window — every column reflects the state as of the snapshot.

In [22]:
rule_inputs = {'freshness_tier', 'impressions_90d', 'avg_position', 'position_tier', 'ctr'}
banned = {'trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d',
          'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'}
print("rule inputs used:", sorted(rule_inputs))
print("overlap with banned columns:", rule_inputs & banned)
assert not (rule_inputs & banned), "leakage: a banned column is being used as a rule input"
print("OK — no banned columns used as rule inputs.")

rule inputs used: ['avg_position', 'ctr', 'freshness_tier', 'impressions_90d', 'position_tier']
overlap with banned columns: set()
OK — no banned columns used as rule inputs.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.